PREPROCESAMIENTO DE DATOS

In [5]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA


In [6]:
df_consolidado = pd.read_csv("data/consolidado.csv")

C:\Users\Usuario\AppData\Local\Temp\ipykernel_17116\2611210249.py:1: DtypeWarning: Columns (0: addrpct, 1: linecm, 2: detailcm) have mixed types. Specify dtype option on import or set low_memory=False.
  df_consolidado = pd.read_csv("data/consolidado.csv")


Identificación del Oficial
Las columnas officrid (ID del oficial), offshld (número de placa) y offverb (nombre del oficial) suelen contener texto o números de identificación. En muchos datasets de la policía de NY, si estos campos están vacíos, significa que el oficial no proporcionó esa información o no fue registrada.
No te interesa cuál es el número de placa (que serían miles de categorías únicas), sino si el oficial se identificó o no.

In [7]:
# 1. Definimos las columnas a limpiar
#columnas_limpiar = ['officrid', 'offshld', 'offverb']

#print("Antes de los cambios:")
#print(df_consolidado[columnas_limpiar].head(6))

# 2. comparación directa: si es un espacio vacío " ", ponemos "N", si no, "Y"
#for col in columnas_limpiar:
    # .strip() elimina espacios accidentales alrededor por si acaso
    #df_consolidado[col] = df_consolidado[col].apply(lambda x: 'N' if str(x).strip() == "" else 'Y')

#print("\nDespués de los cambios:")
#print(df_consolidado[columnas_limpiar].head(6))

#for col in columnas_limpiar:
   # df_consolidado[col] = df_consolidado[col].astype(str) #astype convierte a string, por si acaso hay valores numéricos o nulos que se convirtieron a "N" o "Y"

Identificación del Oficial
Las columnas officrid (ID del oficial), offshld (número de placa) y offverb (nombre del oficial) suelen contener texto o números de identificación. En muchos datasets de la policía de NY, si estos campos están vacíos, significa que el oficial no proporcionó esa información o no fue registrada.
No te interesa cuál es el número de placa (que serían miles de categorías únicas), sino si el oficial se identificó o no.

El grupo cols_u (Imputación de Desconocidos)
Columnas: sector (Sector policial), trhsloc (Ubicación de tránsito), beat (Ruta de patrullaje).

Aquí no quieres perder la información específica (por ejemplo, saber exactamente en qué "sector" ocurrió). Sin embargo, necesitas que los valores vacíos tengan una etiqueta clara ("U") en lugar de ser un espacio en blanco o un valor nulo (NaN).
Esto permite que, al hacer agrupaciones o modelos, los incidentes sin ubicación específica se agrupen en una categoría llamada "U" en lugar de ser ignorados por el algoritmo.

In [8]:
# Definimos los grupos de columnas
# cols_yn = ['officrid', 'offshld', 'offverb']
# cols_u = ['sector', 'trhsloc', 'beat']

# Aplicamos limpieza de Y/N
# for col in cols_yn:
   #  if col in df_consolidado.columns:
   #      df_consolidado[col] = df_consolidado[col].apply(lambda x: 'N' if str(x).strip() == "" else 'Y')
        

# Aplicamos limpieza de 'U'Si el campo está vacío, se le asigna la letra "U" (de Unknown o Unassigned). Si tiene datos, se deja el valor original.
# for col in cols_u:
   #  if col in df_consolidado.columns:
   #      df_consolidado[col] = df_consolidado[col].apply(lambda x: 'U' if str(x).strip() == "" else x)


# for col in (cols_yn + cols_u):
  #   df_consolidado[col] = df_consolidado[col].astype(str)

Identificación del Oficial
Las columnas officrid (ID del oficial), offshld (número de placa) y offverb (nombre del oficial) suelen contener texto o números de identificación. En muchos datasets de la policía de NY, si estos campos están vacíos, significa que el oficial no proporcionó esa información o no fue registrada.
No te interesa cuál es el número de placa (que serían miles de categorías únicas), sino si el oficial se identificó o no.

El grupo cols_u (Imputación de Desconocidos)
Columnas: sector (Sector policial), trhsloc (Ubicación de tránsito), beat (Ruta de patrullaje).

Aquí no quieres perder la información específica (por ejemplo, saber exactamente en qué "sector" ocurrió). Sin embargo, necesitas que los valores vacíos tengan una etiqueta clara ("U") en lugar de ser un espacio en blanco o un valor nulo (NaN).
Esto permite que, al hacer agrupaciones o modelos, los incidentes sin ubicación específica se agrupen en una categoría llamada "U" en lugar de ser ignorados por el algoritmo.

In [9]:
# --- PASO 1: Limpieza y Transformación de Altura ---

# Forzamos que ht_feet y ht_inch sean números.
# 'coerce' convierte cualquier letra o espacio sucio en NaN (vacío) para que no falle la división.
df_consolidado['ht_feet'] = pd.to_numeric(df_consolidado['ht_feet'], errors='coerce')
df_consolidado['ht_inch'] = pd.to_numeric(df_consolidado['ht_inch'], errors='coerce')

# Ahora sí, calculamos los metros (llenando los vacíos temporales con 0 para la suma)Divide las pulgadas en 12 para pasarlos a pies
# suma los pies totales y luego multiplica por 0.3038 para pasarlos a metros.
df_consolidado['meters'] = (df_consolidado['ht_feet'].fillna(0) + (df_consolidado['ht_inch'].fillna(0) / 12)) * 0.3048

# Eliminamos las columnas viejas
df_consolidado.drop(['ht_feet', 'ht_inch'], axis=1, inplace=True)

# --- PASO 2: Estandarización de Etiquetas (Y/N y U) ---
cols_yn = ['officrid', 'offshld', 'offverb']
cols_u = ['sector', 'trhsloc', 'beat']

for col in cols_yn:
    if col in df_consolidado.columns:
        # Limpiamos espacios y convertimos a Y/N
        df_consolidado[col] = df_consolidado[col].astype(str).str.strip().apply(lambda x: 'N' if x == "" or x == "nan" else 'Y')

for col in cols_u:
    if col in df_consolidado.columns:
        # Limpiamos espacios y ponemos 'U' si está vacío
        df_consolidado[col] = df_consolidado[col].astype(str).str.strip().apply(lambda x: 'U' if x == "" or x == "nan" else x)

print("Limpieza previa completada sin errores de tipo.")

Limpieza previa completada sin errores de tipo.


In [10]:
df_consolidado['datestop'] = df_consolidado['datestop'].astype(str) #fecha del arresto, la convertimos a string por si acaso hay valores numéricos o nulos que se convirtieron a "N" o "Y"

# Preparar listas para almacenar los valores de los meses y años
months = []
years = []

# Utilizar un bucle for para procesar cada entrada en la columna datestop
for date in df_consolidado['datestop']:
    # Verificar si la fecha tiene 7 dígitos (mes con un dígito)
    if len(date) == 7: #asegura que la fecha tenga 8 caracteres, si no, le añade un '0' al principio
        
        date = '0' + date

    # Extraer el mes y el año y se convierten a enteros
    month = int(date[0:2]) # slice: toma los primeros dos caracteres del mes
    year = int(date[4:8]) # slice: salta el dìa y toma los ultimos cuatro caracteres (el año)

    # Añadir el mes y año a las listas correspondientes
    months.append(month)
    years.append(year)

# Agregar las listas como columnas nuevas en el DataFrame como variables categoricas y numericas faicles de leer.
df_consolidado['month'] = months
df_consolidado['year'] = years

display(df_consolidado.tail(5))
#eliminamos la columna 'datestop'
df_consolidado = df_consolidado.drop(['datestop'], axis=1)

,Unnamed: 0,year,pct,ser_num,datestop,timestop,recstat,inout,trhsloc,perobs,...,sector,beat,post,xcoord,ycoord,dettypcm,linecm,detailcm,meters,month
11820,449178,2010,60,7062,9262010,45,1,O,H,2.0,...,I,U,,989382,155162,CM,1,20,1.7780,9
11821,362162,2010,75,15797,7302010,10,A,O,P,1.0,...,A,U,,1012757,186018,CM,1,85,1.7018,7
11822,208893,2010,123,830,4302010,1630,A,O,P,3.0,...,E,U,,933868,138600,CM,1,46,1.8288,4
11823,551820,2010,115,13122,11232010,2100,1,O,P,1.0,...,E,U,,1014722,214388,CM,1,85,1.8034,11
11824,6678,2010,14,219,1062010,1406,1,I,T,4.0,...,H,11,,987078,215157,CM,1,23,1.8542,1


In [11]:
# 1. Identificar categóricas (2-99 categorías y tipo string). 
# Si tiene menos de dos valores la columna no sirve para predecir porque no aporta información (Todas las filas dirían lo mismo)
#Si tiene mas de 99 se tendrían muchos valores como nombres de personas o direcciones que no aportan al modelo
categorical_cols = [col for col in df_consolidado.columns
                    if 2 <= df_consolidado[col].nunique() <= 99
                    and df_consolidado[col].dtype == 'str']


# 2. Identificar numéricas clave
numeric_cols = ['month', 'year', 'meters', 'age']


# 3. ELIMINAR DUPLICADOS: Aseguramos que no haya columnas repetidas entre las numericas y las categoricas
# Si una columna está en ambas, la dejamos solo en una (usualmente la numérica)
categorical_cols = [c for c in categorical_cols if c not in numeric_cols]


# 4. Crear df_copy_18 asegurando columnas ÚNICAS
selected_columns = list(dict.fromkeys(categorical_cols + numeric_cols)) # Elimina duplicados manteniendo orden
df_copy_18 = df_consolidado[selected_columns].copy()

# 5. Definir las listas para el Pipeline (Sin duplicados)
variables_categoricas = categorical_cols
variables_numericas = numeric_cols

# Convertir a string las categóricas para evitar el error de tipos mezclados para evitar fallas en el pipeline de proprocesamiento al encontrarse con columnas mixtas
for col in variables_categoricas:
    df_copy_18[col] = df_copy_18[col].astype(str)

print(f"Selección lista. Columnas únicas: {len(df_copy_18.columns)}")

Selección lista. Columnas únicas: 77


In [12]:
df_copy_18.to_csv("data/df_copy_18.csv", index=False)

CONSTRUCCIÓN DEL PIPELINE
Su función es automatizar la transformación de los datos crudos en una matriz numérica optimizada que un algoritmo pueda entender.
Es decir, "preparar los datos antes de trabajarlos". 


In [13]:

preprocessor = ColumnTransformer(  #Organiza las transformaciones por tipo de variable (no se puede tratar igual edad y raza)
    transformers=[
        # Sub-pipeline para Números
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')), # Técnica de imputación con la media por si queda algun valor vacio
            ('scaler', StandardScaler())                  # Ajusta todos los numeros par que tengan media 0 y varianza 1
        ]), variables_numericas),

        # Sub-pipeline para Categorías
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')), # Imputación de categorías faltantes con 'missing'
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # Codificación One-Hot para variables categóricas, crea una columna nueva de ceros por cada categoria
        ]), variables_categoricas)
    ])
# El PCA comprime toda esa información en unas pocas columnas nuevas (componentes).
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)) # Reducción de la cantidad de columnas lo mas que puede,conserva el 95% de la informaciòn (varianza original) 
])


X_final = final_pipeline.fit_transform(df_copy_18) #fit_transform ejecuta el pipeline completo: genera matriz de numeros lista para modelar, con las columnas originales transformadas y reducidas a unas pocas componentes principales que conservan la mayor parte de la información original.

print("Pipeline ejecutado con éxito.")
print(f"Forma de la matriz final: {X_final.shape}")

Pipeline ejecutado con éxito.
Forma de la matriz final: (11825, 69)
